# Build: integrated oligodendrocyte atlas across the whole-transcriptome datasets

Pulls every cell / nucleus labelled `Oligodendrocyte` (see `docs/cell_type_annotation.md`) from the twelve single-cell / single-nucleus datasets that deposit raw counts, maps human genes to mouse symbols with one-to-one MGI orthologs, and integrates them with scVI (batch = dataset) and scANVI (labels = Lerma-Martin's oligodendrocyte subtypes `OL_Homeo*` / `OL_Dis*`, everything else unlabelled). The atlas is the substrate for `analysis_oligodendrocyte_atlas.ipynb`.

Included: Park 2023, aging snRNA-seq HIP/CP, Ximerakis 2019, Kaya 2022, Zhou 2020, LPC + cuprizone, Serpina3n-cKO cuprizone, Jäkel 2019, Absinta 2021, Leng 2021, Sadick 2022, Lerma-Martin 2024. Excluded: the spatial datasets (spots, not cells), Schirmer 2019 (UCSC matrix is log-normalised, no counts), Falcão 2018 (Smart-seq2 normalised values, no counts) and the targeted Xenium panels. Per dataset at most 20,000 oligodendrocytes are used (random subsample, seed 0).

Requires the `sc_py312` environment (scvi-tools ≥ 1.4). Output: `<OLIGOC4B_PUBLIC_PROCESSED_DIR>/oligodendrocyte_atlas.h5ad`.

In [ ]:
import os, sys, gc, warnings, time
warnings.filterwarnings("ignore")
sys.path.insert(0, "../../scripts")
import numpy as np, pandas as pd, scipy.sparse as sp, scanpy as sc, anndata as ad
import oligoc4b_public as pub
import scvi, torch
sc.settings.verbosity = 1
scvi.settings.seed = 0
np.random.seed(0)
print("scvi", scvi.__version__, "| torch", torch.__version__, "| mps:", torch.backends.mps.is_available())

MAX_PER_DATASET = 20000
EXCLUDE = {"Chen2020_ST_AppNLGF_mouse", "LermaMartin2024_MS_human_Visium", "SenescentGlia2025_MS_human_Visium", "Schirmer2019_MS_human_snRNA"}
OUT = os.path.join(pub.PROCESSED_DIR, "oligodendrocyte_atlas.h5ad")
h2m = pub.load_orthologs()
print(f"orthologs: {len(h2m)} human->mouse one-to-one pairs from {pub.ORTHOLOG_TABLE}")

In [ ]:
parts = []
for name, path in pub.processed_paths().items():
    if name in EXCLUDE or name == "oligodendrocyte_atlas":
        continue
    a = sc.read_h5ad(path, backed="r")
    mask = (a.obs["cell_type_coarse"].astype(str) == "Oligodendrocyte").values
    idx = np.where(mask)[0]
    if len(idx) > MAX_PER_DATASET:
        idx = np.sort(np.random.RandomState(0).choice(idx, MAX_PER_DATASET, replace=False))
    ol = a[idx].to_memory()
    if "counts" not in ol.layers:
        print(f"[skip] {name}: no raw counts"); continue
    X = ol.layers["counts"]
    b = ad.AnnData(X=sp.csr_matrix(X), obs=ol.obs.copy(), var=pd.DataFrame(index=ol.var_names))
    if b.obs["species"].iloc[0] == "human":
        b = pub.to_mouse_symbols(b, h2m)
    # keep a compact, harmonised obs
    keep_cols = ["dataset", "species", "modality", "sample", "group", "group_ref", "cell_type_original", "condition_original",
                 "genotype", "trem2", "fad", "region", "age_months", "age_group", "braak", "control_type", "treatment", "diet", "subtype"]
    b.obs = b.obs[[c for c in keep_cols if c in b.obs.columns]].copy()
    for c in keep_cols:                       # same obs columns in every part (ad.concat keeps only shared columns)
        if c not in b.obs.columns:
            b.obs[c] = "NA"
        b.obs[c] = b.obs[c].astype(str)
    b.obs = b.obs[keep_cols]
    b.obs_names = [f"{name}:{o}" for o in b.obs_names]
    parts.append(b)
    print(f"{name:42s} {b.n_obs:6d} oligodendrocytes, {b.n_vars:6d} genes ({'mapped to mouse symbols' if b.obs['species'].iloc[0]=='human' else 'mouse'})")
    del a, ol; gc.collect()

# genes shared by all datasets
shared = set(parts[0].var_names)
for b in parts[1:]:
    shared &= set(b.var_names)
shared = sorted(shared)
print(f"\nshared genes across {len(parts)} datasets: {len(shared)}")
atlas = ad.concat([b[:, shared].copy() for b in parts], join="inner", index_unique=None)
atlas.layers["counts"] = atlas.X.copy()
del parts; gc.collect()
print(atlas)

In [ ]:
# normalise for downstream scoring / plotting; HVGs on counts (seurat_v3) within dataset
sc.pp.normalize_total(atlas, target_sum=1e4)
sc.pp.log1p(atlas)
sc.pp.highly_variable_genes(atlas, flavor="seurat_v3", layer="counts", n_top_genes=3000, batch_key="dataset", subset=False)
# make sure the genes we care about are carried in the model
for g in ["C4b", "C4a", "Serpina3n", "H2-D1", "H2-K1", "B2m", "Cd59a", "Cr1l", "C1qa", "C3", "Hc", "C5ar1", "Cfb", "Klk6", "Apod", "Trf", "Cd9", "Cldn11", "Plp1", "Mbp"]:
    if g in atlas.var_names:
        atlas.var.loc[g, "highly_variable"] = True
print("HVGs:", int(atlas.var["highly_variable"].sum()))

In [ ]:
# scVI (batch = dataset) then scANVI seeded with Lerma-Martin oligodendrocyte subtypes
atlas.obs["scanvi_label"] = "Unknown"
lm = atlas.obs["dataset"] == "LermaMartin2024_MS_human_snRNA"
sub = atlas.obs.loc[lm, "subtype"].astype(str) if "subtype" in atlas.obs else atlas.obs.loc[lm, "cell_type_original"].astype(str)
sub = sub.where(sub.str.startswith("OL_") & ~sub.isin(["OL_NA"]), "Unknown")
atlas.obs.loc[lm, "scanvi_label"] = sub.values
print(atlas.obs["scanvi_label"].value_counts())

hv = atlas[:, atlas.var["highly_variable"]].copy()
scvi.model.SCVI.setup_anndata(hv, layer="counts", batch_key="dataset")
model = scvi.model.SCVI(hv, n_latent=30, n_layers=2, gene_likelihood="nb")
accel = "cpu"
try:
    if torch.backends.mps.is_available():
        accel = "mps"
except Exception:
    pass
t0 = time.time()
try:
    model.train(max_epochs=None, accelerator=accel, early_stopping=True, batch_size=512)
except Exception as exc:
    print("training on", accel, "failed:", type(exc).__name__, exc, "-> retrying on cpu")
    model = scvi.model.SCVI(hv, n_latent=30, n_layers=2, gene_likelihood="nb")
    model.train(max_epochs=None, accelerator="cpu", early_stopping=True, batch_size=512)
print(f"scVI trained in {(time.time()-t0)/60:.1f} min on {accel}; epochs run: {len(model.history['elbo_train'])}")
atlas.obsm["X_scVI"] = model.get_latent_representation()

In [ ]:
t0 = time.time()
lvae = scvi.model.SCANVI.from_scvi_model(model, unlabeled_category="Unknown", labels_key="scanvi_label")
try:
    lvae.train(max_epochs=20, n_samples_per_label=200, accelerator=accel, batch_size=512)
except Exception as exc:
    print("scANVI on", accel, "failed:", type(exc).__name__, "-> cpu")
    lvae = scvi.model.SCANVI.from_scvi_model(model, unlabeled_category="Unknown", labels_key="scanvi_label")
    lvae.train(max_epochs=20, n_samples_per_label=200, accelerator="cpu", batch_size=512)
print(f"scANVI trained in {(time.time()-t0)/60:.1f} min")
atlas.obsm["X_scANVI"] = lvae.get_latent_representation()
atlas.obs["ol_subtype_pred"] = lvae.predict()
soft = lvae.predict(soft=True)
atlas.obs["ol_subtype_pred_conf"] = soft.max(axis=1).values
atlas.obs["ol_dis_prob"] = soft[[c for c in soft.columns if c.startswith("OL_Dis")]].sum(axis=1).values
print(pd.crosstab(atlas.obs["dataset"], atlas.obs["ol_subtype_pred"]))

In [ ]:
sc.pp.neighbors(atlas, use_rep="X_scANVI", n_neighbors=15)
sc.tl.leiden(atlas, resolution=0.5, key_added="leiden_0.5", flavor="igraph", n_iterations=2, directed=False)
sc.tl.leiden(atlas, resolution=1.0, key_added="leiden_1.0", flavor="igraph", n_iterations=2, directed=False)
sc.tl.umap(atlas, min_dist=0.3)
# gene-program scores used downstream
def score(genes, name):
    genes = [g for g in genes if g in atlas.var_names]
    sc.tl.score_genes(atlas, genes, score_name=name, use_raw=False)
score(["H2-D1", "H2-K1", "B2m", "Ifi27", "Ifi27l2a", "Irf9", "Stat3", "Psmb8", "Nlrc5", "Irgm1", "Bst2", "Ifit3", "Ifitm3"], "score_MHCI_IFN")
score(["Serpina3n", "Klk6", "Apod", "Trf", "Cd9", "Cldn11", "Gfap", "Vim", "Cryab", "Il33"], "score_C4b_program")
score(["Cd59a", "Cr1l", "Cd55", "Cfh"], "score_complement_regulators")
score(["Mbp", "Plp1", "Mog", "Mag", "Cnp", "Mobp"], "score_myelin")
x = atlas[:, "C4b"].X
atlas.obs["C4b_log"] = np.asarray(x.todense()).ravel() if sp.issparse(x) else np.asarray(x).ravel()
atlas.obs["C4b_pos"] = atlas.obs["C4b_log"] > 0
print(atlas)

In [ ]:
# per-dataset NMF programs (for cross-dataset program matching in the analysis notebook)
from sklearn.decomposition import NMF
K, N_CELLS = 12, 8000
hv_genes = atlas.var_names[atlas.var["highly_variable"].values]
prog_rows = []
for name in atlas.obs["dataset"].unique():
    idx = np.where(atlas.obs["dataset"].values == name)[0]
    if len(idx) < 500:
        continue
    if len(idx) > N_CELLS:
        idx = np.random.RandomState(0).choice(idx, N_CELLS, replace=False)
    X = atlas[idx, hv_genes].X
    X = X.toarray() if sp.issparse(X) else np.asarray(X)
    keep = X.sum(axis=0) > 0
    nmf = NMF(n_components=K, init="nndsvda", max_iter=400, random_state=0)
    W = nmf.fit_transform(X[:, keep])
    H = pd.DataFrame(nmf.components_, columns=hv_genes[keep])
    H = H.div(H.sum(axis=1), axis=0)              # gene loadings sum to 1 per program
    for k in range(K):
        prog_rows.append(pd.Series(H.iloc[k], name=f"{name}|P{k}"))
    print(f"{name}: NMF on {len(idx)} cells x {keep.sum()} genes done")
PROGRAMS = pd.DataFrame(prog_rows).fillna(0.0)
atlas.uns["nmf_programs"] = PROGRAMS.to_dict()
PROGRAMS.to_csv(os.path.join(pub.PROCESSED_DIR, "oligodendrocyte_atlas_nmf_programs.csv"))
print(PROGRAMS.shape)

In [ ]:
atlas.write(OUT)
print("wrote", OUT, atlas.shape)
print(atlas.obs.groupby("dataset", observed=True).size())